Project : DeepResearch (AI Agents) / Proyecto: DeepResearch (Agentes de IA)

📦 Initial Setup and Libraries / Configuración Inicial y Bibliotecas
Goal: Load required dependencies and set up the environment.
Objetivo: Cargar las dependencias necesarias y configurar el entorno.

In [ ]:
# [EN] Import essential libraries for the project  
# [ES] Importamos bibliotecas esenciales para el proyecto  
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool  # AI Agents framework / Framework de agentes de IA  
from agents.model_settings import ModelSettings  # Model settings / Configuración de modelos  
from pydantic import BaseModel, Field  # Data validation / Validación de datos  
from dotenv import load_dotenv  # Environment variables / Variables de entorno  
import asyncio  # Asynchronous programming / Programación asíncrona  
import sendgrid  # Email sending / Envío de emails  
import os  # OS interaction / Interacción con el sistema operativo  
from sendgrid.helpers.mail import Mail, Email, To, Content  # Email tools / Herramientas para emails  
from typing import Dict  # Data typing / Tipado de datos  
from IPython.display import display, Markdown  # Notebook display / Visualización en notebooks  
import gradio as gr  # Web UI / Interfaz web  
import nest_asyncio  # Asyncio support in Jupyter / Soporte para asyncio en Jupyter  

# [EN] Load environment variables (e.g., API keys) from .env file  
# [ES] Cargamos variables de entorno (ej: API keys) desde un archivo .env  
load_dotenv(override=True)  

🔍 Web Search Agent / Agente de Búsqueda Web
Goal: Create an agent that performs web searches and summarizes results.
Objetivo: Crear un agente que realice búsquedas en la web y resuma los resultados.

In [ ]:
# [EN] Instructions for the search agent:  
# - Search the web and generate concise summaries (2-3 paragraphs, <300 words).  
# [ES] Instrucciones para el agente de búsqueda:  
# - Busca términos en la web y genera resúmenes concisos (2-3 párrafos, <300 palabras).  
instructions = "Eres asistente de investigación. Dado un término de búsqueda, \\\nlo buscas en la web y elaboras un resumen conciso de los resultados.\\\nEl resumen debe tener de 2 a 3 párrafos y menos de 300 palabras. \\\nCapta los puntos principales. Escribe de forma concisa. Y no más de 200 palabras"  

# [EN] Agent setup with:  
# - Name, instructions, web search tool, and GPT-4 model.  
# [ES] Configuramos el agente con:  
# - Nombre, instrucciones, herramienta de búsqueda y modelo GPT-4.  
search_agent = Agent(  
    name="Search Agent",  
    instructions=instructions,  
    tools=[WebSearchTool(search_context_size='low')],  # Search tool / Herramienta de búsqueda  
    model="gpt-4o-mini",  # Language model / Modelo de lenguaje  
    model_settings=ModelSettings(tool_choice="required")  # Forces tool usage / Fuerza el uso de herramientas  
)  

# [EN] Example search: "Top 3 AI frameworks in 2025"  
# [ES] Ejemplo de búsqueda: "Los 3 frameworks de IA más populares en 2025"  
message = "los 3 frameworks de IA más recientes y populares de 2025"  
with trace("Buscador ejemplo"):  # Logs trace to OpenAI / Registra la traza en OpenAI  
    result = await Runner.run(search_agent, message)  
    display(Markdown(result.final_output))  # Displays result in Markdown / Muestra el resultado en Markdown  

📝 Search Planning Agent / Agente de Planificación de Búsquedas
Goal: Generate optimized search terms for a query.
Objetivo: Generar términos de búsqueda optimizados para una consulta.

In [ ]:
# [EN] Define how many searches to perform and planner agent instructions.  
# [ES] Definimos cuántas búsquedas realizar y las instrucciones para el agente planificador.  
HOW_MANY_SEARCHES = 3  
INSTRUCTION = f"Eres un asistente de investigación. Debes generar {HOW_MANY_SEARCHES} términos de búsqueda para responder a una pregunta."  

# [EN] Pydantic models for output structure:  
# - WebSearchItem: Reason and search query.  
# - WebSearchPlan: List of planned searches.  
# [ES] Modelos Pydantic para estructurar los datos de salida:  
# - WebSearchItem: Razón y término de búsqueda.  
# - WebSearchPlan: Lista de búsquedas planificadas.  
class WebSearchItem(BaseModel):  
    reason: str = Field(description="Una razón por la que la búsqueda es útil")  
    query: str = Field(description="El término de búsqueda en sí.")  

class WebSearchPlan(BaseModel):  
    searches: list[WebSearchItem] = Field(description="Una lista de las búsquedas para mostrar")  

# [EN] Create planner agent with structured output (WebSearchPlan).  
# [ES] Creamos el agente planificador con salida estructurada (WebSearchPlan).  
planner_agent = Agent(  
    name="planner_agent",  
    instructions=INSTRUCTION,  
    model="gpt-4o-mini",  
    output_type=WebSearchPlan  # Defined output type / Tipo de salida definido  
)  

# [EN] Example: Plan searches for the same query.  
# [ES] Ejemplo: Planificar búsquedas para la misma consulta.  
message = "los 3 frameworks de IA más recientes y populares de 2025"  
with trace("buscador ejemplo"):  
    result = await Runner.run(planner_agent, message)  
    print(result.final_output)  # Displays search plan / Muestra el plan de búsqueda  

✉️ Email Sending Agent / Agente de Envío de Emails
Goal: Send reports via email using SendGrid.
Objetivo: Enviar informes por email usando SendGrid.

In [ ]:
# [EN] Function to send emails (using SendGrid API).  
# [ES] Función para enviar emails (usando SendGrid API).  
@function_tool  
def send_email(subject: str, html_body: str) -> Dict[str, str]:  
    """ Send out an email with the given subject and HTML body """  
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))  # API key / Clave API  
    from_email = Email("testingavomo@gmal.com")  # Sender email / Email remitente  
    to_email = To("contactola24podcast@gmail.com")  # Receiver email / Email destinatario  
    content = Content("text/html", html_body)  # HTML content / Contenido HTML  
    mail = Mail(from_email, to_email, subject, content).get()  # Email setup / Configura el email  
    response = sg.client.mail.send.post(request_body=mail)  # Sends email / Envía el email  
    return {"status": "success"}  

# [EN] Agent specialized in sending HTML-formatted reports via email.  
# [ES] Agente especializado en enviar informes en formato HTML por email.  
instructions = """  
    Puede enviar un correo electrónico HTML con un formato atractivo basado en un informe detallado.  
    Se le proporcionará un informe detallado. Debe usar su tool send_email,  
    proporcionando el informe convertido en HTML limpio y bien presentado, con un asunto adecuado.  
"""  

email_agent = Agent(  
    name="email_agent",  
    instructions=instructions,  
    tools=[send_email],  # Email sending tool / Herramienta de envío de emails  
    model="gpt-4o-mini"  
)  

📄 Report Generation Agent / Agente de Generación de Informes
Goal: Create detailed reports in Markdown (5-10 pages).
Objetivo: Crear informes detallados en Markdown (5-10 páginas).

In [ ]:
# [EN] Writer agent instructions:  
# - First draft an outline, then generate a detailed Markdown report.  
# [ES] Instrucciones para el agente escritor:  
# - Genera un esquema primero, luego un informe extenso en Markdown.  
INSTRUCTIONS = """  
    Eres un investigador sénior encargado de redactar un informe coherente para una consulta de investigación.  
    Se te proporcionará la consulta original y un asistente de investigación realizará una investigación inicial.  
    Primero, debes elaborar un esquema para el informe que describa la estructura y el flujo del mismo.  
    Luego, genera el informe y devuélvelo como resultado final.  
    El resultado final debe estar en formato Markdown y debe ser extenso y detallado (1000+ palabras).  
"""  

# [EN] Report structure (using Pydantic):  
# - Short summary, Markdown report, follow-up questions.  
# [ES] Estructura del informe (usando Pydantic):  
# - Resumen corto, informe en Markdown, preguntas de seguimiento.  
class ReportData(BaseModel):  
    short_summary: str = Field(description="Un breve resumen de 2 a 3 oraciones de los hallazgos.")  
    markdown_report: str = Field(description="El informe final.")  
    follow_up_questions: str = Field(description="Temas sugeridos para investigar más a fondo.")  

writer_agent = Agent(  
    name="writer agent",  
    instructions=INSTRUCTIONS,  
    model="gpt-4o-mini",  
    output_type=ReportData  # Structured output / Salida estructurada  
)  

🔄 Automated Research Pipeline / Pipeline de Investigación Automatizado
Goal: Connect all agents into an automated workflow.
Objetivo: Conectar todos los agentes en un flujo de trabajo automatizado.

In [ ]:
# [EN] Function to perform a single search using the search agent.  
# [ES] Función para realizar una búsqueda individual con el agente de búsqueda.  
async def search(item: WebSearchItem):  
    print("Iniciando busqueda...")  
    input = f"Query: {item.query}\\n Reason: {item.reason}"  
    result = await Runner.run(search_agent, input)  
    return result  

# [EN] Function to plan searches using the planner agent.  
# [ES] Función para planificar búsquedas con el agente planificador.  
async def plan_search(query: str):  
    print("Buscando plan...")  
    result = await Runner.run(planner_agent, f"query: {query}")  
    print(f"Vamos a generar {len(result.final_output.searches)} búsquedas")  
    return result.final_output  

# [EN] Execute all planned searches in parallel.  
# [ES] Ejecuta todas las búsquedas planificadas en paralelo.  
async def perform_search(search_plan: WebSearchPlan):  
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]  
    return await asyncio.gather(*tasks)  

# [EN] Generate a detailed report from search results.  
# [ES] Genera un informe detallado a partir de los resultados.  
async def write_report(query: str, search_results: list[str]):  
    input = f"original query: {query}\\n list of search results: {search_results}"  
    result = await Runner.run(writer_agent, input)  
    print("Reporte finalizado")  
    return result.final_output  

# [EN] Send the report via email.  
# [ES] Envía el informe por email.  
async def send_email(report: ReportData):  
    result = await Runner.run(email_agent, report.markdown_report)  
    print("Email enviado")  
    return report  


# [EN] Full pipeline example:  
# 1. Plan → 2. Search → 3. Write → 4. Email.  
# [ES] Ejemplo completo:  
# 1. Planificar → 2. Buscar → 3. Escribir → 4. Enviar email.  
query = "Los 3 frameworks de Agentes de IA más importantes/populares de 2025."  
search_plan = await plan_search(query)  
results = await perform_search(search_plan)  
report = await write_report(query, results)  
email = await send_email(report)  
print('¡Eureka!')  # 🎉  

🌐 Web Interface with Gradio / Interfaz Web con Gradio
Goal: Create an interactive UI for running research queries.
Objetivo: Crear una interfaz interactiva para ejecutar consultas de investigación.

In [ ]:

async def send_report(report: ReportData) -> dict:
    """[EN] Send the report via email
       [ES] Envía el informe por correo electrónico"""
    print("✉️ Enviando email... / Sending email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("📤 Email enviado / Email sent")
    return result

async def full_research_pipeline(query: str) -> ReportData:
    """[EN] Complete research workflow
       [ES] Flujo completo de investigación"""
    search_plan = await plan_search(query)
    search_results = await perform_search(search_plan)
    report = await write_report(query, search_results)
    await send_report(report)
    return report

# ====================
# INTERFAZ GRADIO / GRADIO INTERFACE
# ====================

def setup_gradio_interface():
    """[EN] Configure and launch Gradio interface
       [ES] Configura y lanza la interfaz Gradio"""
    
    # Aplicar nest_asyncio para Jupyter/Colab
    nest_asyncio.apply()
    
    async def gradio_chat(message, history):
        """[EN] Chat function for Gradio
           [ES] Función de chat para Gradio"""
        try:
            research = await full_research_pipeline(message)
            return f"""
            ## 🔍 Reporte: {message}
            {research.markdown_report}
            ### ✨ Resumen:
            {research.short_summary}
            """
        except Exception as e:
            return f"❌ Error: {str(e)}"
    
    # Configurar interfaz
    interface = gr.ChatInterface(
        fn=gradio_chat,
        title="🧠 DeepResearch AI",
        description="""[EN] Enter a research topic (e.g. 'Top AI frameworks 2025')
                    [ES] Ingresa un tema de investigación (ej. 'Mejores frameworks IA 2025')""",
        examples=[
            "Frameworks de IA más populares 2025",
            "Avances en inteligencia artificial 2025",
            "Nuevos lenguajes de programación emergentes"
        ],
        theme="soft"
    )
    
    # Lanzar interfaz/Launch Interface
    interface.launch(
        server_name="0.0.0.0",
        server_port=7862,
        share=False,
        inbrowser=True
    )

# ====================
# EJECUCIÓN / EXECUTION
# ====================

if __name__ == "__main__":
    print("🚀 Iniciando DeepResearch Pipeline... / Starting DeepResearch Pipeline...")
    setup_gradio_interface()